# DMOSpeech 2 — EXP-031: three single-formula handovers against the previous methods

- **Task ID:** `EVAL-024` (development experiment, preregistered before any synthesis; Amendment 1 added `QH3` and `BTH3`).
- **Methods under test (applied to every input, no routing; 12 passes each):** DMOSpeech 2's teacher runs 3 steps and the student finishes, but only part of the teacher crosses:
  - `TOH3`: the teacher's timing, as a time map of the student's own draft;
  - `QH3` (Quantile Handover): in every mel band, the student's values placed in the teacher's time order;
  - `BTH3` (Blind-Teacher Handover): the timing of a teacher that never receives the prompt (target text only).
- **Compared on the same cases with:** `W4` (DMOSpeech 2's default), `W8` (reference), `STAC`, Chapter 3's SNR + drift policy (`CH3`, and `CH3_LTT` with the certified threshold), and `RND`, a teacher-free random re-timing that tests whether the teacher is needed. The SNR-routed versions (`SNR_TOH`, `SNR_W4`, `SNR_RND`) are exploratory.
- **Synthesized arms (9):** `STU`, `W4`, `W5`, `W6`, `W8`, `TOH3`, `QH3`, `BTH3`, `RND`; the routed methods are read from them.
- **Cases:** development speakers 1-30 (never run through TOH) × clean, 25 dB and 15 dB white noise, reverberation (0.3 s, 0.9 s) and 15 dB cafeteria babble (180 cases), plus speakers 1-20 × 12 seeds for clean pitch variety.
- **Estimated T4 runtime:** about 3 hours with `STAGE = "all"` (the default). It resumes from Drive if the runtime disconnects: just **Run all** again.
- **When it finishes,** the results zip is saved to Drive (`MyDrive/exp031_snr_toh_method_comparison/`) and also downloaded.


In [ ]:
import os
import subprocess

# GPU checked with nvidia-smi, not PyTorch: the notebook kernel then holds no PyTorch or CUDA state in RAM,
# so the runner (two TTS models, Whisper large-v3, WavLM and UTMOS) has all of it. Results are unaffected.
q = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), "No GPU found. Switch the runtime to T4 GPU (Runtime > Change runtime type)."
gpu_name = q.stdout.strip().splitlines()[0]
print("GPU:", gpu_name)
assert "T4" in gpu_name, f"Expected a Tesla T4 (found {gpu_name}): the smoke test reproduces EXP-027's T4 scores."

from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/results"
DRIVE_DIR = "/content/drive/MyDrive/exp031_snr_toh_method_comparison"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("Drive directories ready:", DRIVE_DIR)

## 1. Clone Repositories, Install Dependencies & Verify Preregistration

Clones `Shonget-DMOSpeech2` (pinned commit `8a7868fc...`) and `Final-Thesis-Setup` (branch `rhnl-3`), which holds `code/colab/run_method_comparison.py` and `research/experiments/EXP-031-snr-toh-method-comparison/preregistration.json`.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_BRANCH = "rhnl-3"
DMO_COMMIT = "8a7868fc3345702a586173447f4869205b501f26"
DMO_REPO = "https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git"
THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
STAGE = "all"  # runs quality (180 cases) then diversity (20 x 12); resumes from Drive. Other options: "quality" or "diversity"

token = ""
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass
thesis_url = f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git"

if not os.path.exists(DMO_ROOT):
    subprocess.run(["git", "clone", DMO_REPO, DMO_ROOT], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", DMO_ROOT, "checkout", "-q", DMO_COMMIT], check=True)

if not os.path.exists(THESIS_ROOT):
    subprocess.run(["git", "clone", "-b", THESIS_BRANCH, thesis_url, THESIS_ROOT], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "fetch", "-q", "origin", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "checkout", "-q", THESIS_BRANCH], check=True)
subprocess.run(["git", "-C", THESIS_ROOT, "pull", "-q", "origin", THESIS_BRANCH], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy==2.2.3", "pandas==2.2.3"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{DMO_ROOT}/requirements.txt"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torchdiffeq", "vocos", "speechbrain", "transformers", "jiwer",
    "soundfile", "librosa", "pyarrow", "scipy", "jieba", "accelerate", "huggingface_hub"
], check=True)

prereg_path = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/preregistration.json"
assert os.path.exists(prereg_path), f"Missing {prereg_path} on branch {THESIS_BRANCH}"
prereg = json.loads(open(prereg_path, encoding="utf-8").read())
print(f"Preregistration verified: sha256={prereg['sha256'][:12]}..., qual={prereg['n_qual_cases']}, div={prereg['n_div_speakers']}x{prereg['div_seeds']}")

## 2. Checkpoints & Seed-TTS English Audio (Drive Cache Supported)


In [ ]:
import hashlib
import os
import shutil
import subprocess

DMO_ROOT = "/content/Shonget-DMOSpeech2"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
os.makedirs(f"{DMO_ROOT}/ckpts", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/ckpts", exist_ok=True)

CKPTS = {
    "model_85000.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt",
        "e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c",
    ),
    "model_1500.pt": (
        "https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt",
        "6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2",
    ),
}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

for name, (url, expected) in CKPTS.items():
    local_p = f"{DMO_ROOT}/ckpts/{name}"
    cache_p = f"{DRIVE_CACHE}/ckpts/{name}"
    if not os.path.exists(local_p) and os.path.exists(cache_p):
        shutil.copy2(cache_p, local_p)
    if not os.path.exists(local_p):
        subprocess.run(["wget", "-c", url, "-O", local_p], check=True)
    digest = sha256_file(local_p)
    assert digest == expected, f"{name} SHA-256 mismatch: got {digest}"
    if not os.path.exists(cache_p):
        shutil.copy2(local_p, cache_p)
    print(f"{name}: SHA-256 verified")

data_dir = f"{DMO_ROOT}/data/seed-tts-eval"
probe_wav = f"{data_dir}/en/prompt-wavs/common_voice_en_10933823.wav"
cache_data = f"{DRIVE_CACHE}/seed-tts-eval"
if (not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000) and os.path.exists(f"{cache_data}/en/meta.lst"):
    shutil.copytree(cache_data, data_dir, dirs_exist_ok=True)
if not os.path.exists(probe_wav) or os.path.getsize(probe_wav) < 1000:
    from huggingface_hub import snapshot_download
    shutil.rmtree(data_dir, ignore_errors=True)
    snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir=data_dir)
assert os.path.exists(probe_wav) and os.path.getsize(probe_wav) >= 1000, "Seed-TTS audio missing"
print("Seed-TTS audio verified [OK]")

## 3. Cafeteria Babble Noise (DEMAND, Drive Cache Supported)

The `babble15` condition mixes DEMAND's cafeteria recording (`PCAFETER/ch01.wav`, Thiemann, Ito and Vincent, 2013) into the prompt at 15 dB, as in EXP-011. The archive is cached in `MyDrive/dmo_cache/demand/` after the first download.


In [ ]:
import hashlib
import os
import shutil
import subprocess

DEMAND = "/content/demand"
DRIVE_CACHE = "/content/drive/MyDrive/dmo_cache"
ZIP_URL = "https://zenodo.org/records/1227121/files/PCAFETER_16k.zip?download=1"
LOCAL_ZIP = "/content/PCAFETER_16k.zip"
CACHE_ZIP = f"{DRIVE_CACHE}/demand/PCAFETER_16k.zip"
WAV = f"{DEMAND}/PCAFETER/ch01.wav"

if not os.path.exists(WAV):
    os.makedirs(DEMAND, exist_ok=True)
    if not os.path.exists(LOCAL_ZIP) and os.path.exists(CACHE_ZIP):
        shutil.copy2(CACHE_ZIP, LOCAL_ZIP)
    if not os.path.exists(LOCAL_ZIP):
        subprocess.run(["wget", "-q", "-c", "-O", LOCAL_ZIP, ZIP_URL], check=True)
    subprocess.run(["unzip", "-q", "-o", LOCAL_ZIP, "-d", DEMAND], check=True)
    if not os.path.exists(CACHE_ZIP):
        os.makedirs(os.path.dirname(CACHE_ZIP), exist_ok=True)
        shutil.copy2(LOCAL_ZIP, CACHE_ZIP)
if not os.path.exists(WAV):
    print(subprocess.run(["find", DEMAND, "-maxdepth", "3"], capture_output=True, text=True).stdout[:2000])
    raise FileNotFoundError("PCAFETER/ch01.wav not found: check the archive layout printed above")
with open(WAV, "rb") as f:
    digest = hashlib.sha256(f.read()).hexdigest()
print(f"DEMAND babble track ready: {WAV} ({os.path.getsize(WAV)} bytes, sha256 {digest[:12]}...)")


## 4. Restore Any Partial Results from Google Drive


In [ ]:
import csv
import os
import shutil

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/results"
DRIVE_DIR = "/content/drive/MyDrive/exp031_snr_toh_method_comparison"
os.makedirs(OUT_DIR, exist_ok=True)

for rel in ("quality.csv", "diversity.csv", "smoke/quality.csv", "smoke/diversity.csv", "smoke/smoke_check.json"):
    src = os.path.join(DRIVE_DIR, rel)
    dst = os.path.join(OUT_DIR, rel)
    if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print("Restored from Drive:", rel)

for fname, target in (("quality.csv", 180), ("diversity.csv", 20)):
    p = os.path.join(OUT_DIR, fname)
    n = sum(1 for _ in csv.DictReader(open(p, encoding="utf-8"))) if os.path.exists(p) else 0
    print(f"{fname}: {n} / {target} rows")

## 5. Pre-Flight Smoke Test

Verifies the preregistration hashes and exact identities (all `<= 1e-5`): the split student loop, the identity time map, TOH with the identity map equal to the student alone, the Quantile Handover of the draft onto itself equal to the draft, the Quantile Handover's values equal to the draft's values in every band, and the blind teacher giving the same payload twice. It also checks TOH3's, BTH3's and the random map on real teacher runs (monotone, slope-limited, inside the band), the three policies' step choices, reproduces EXP-027's `W4` scores (`<= 0.01`), and runs a micro version of both stages, including one babble prompt.

If it stops with exit code -9 (`died with <Signals.SIGKILL: 9>`), the machine ran out of RAM: use **Runtime > Disconnect and delete runtime**, then **Run all**.


In [ ]:
import json
import os
import subprocess
import sys

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/results"
RUNNER = f"{THESIS_ROOT}/code/colab/run_method_comparison.py"
DEMAND = "/content/demand"

def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {"MALLOC_ARENA_MAX": "2", "PYTHONUNBUFFERED": "1"}  # fewer allocator arenas while the large models load; results are unaffected

os.chdir(THESIS_ROOT)
proc = subprocess.Popen([sys.executable, "-u", RUNNER, "smoke", "--dmo_root", DMO_ROOT, "--demand_dir", DEMAND, "--out_dir", OUT_DIR],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
for line in proc.stdout:
    print(line, end="", flush=True)
code = proc.wait()
if code in (-9, 137):
    raise SystemExit("The smoke test was killed (exit code -9): the machine ran out of RAM. "
                     "Use Runtime > Disconnect and delete runtime, then Run all again.")
assert code == 0, f"Smoke test failed with code {code}"
print(json.dumps(json.loads(open(f"{OUT_DIR}/smoke/smoke_check.json", encoding="utf-8").read()), indent=2))

## 6. Run Synthesis (`all` = about 3 hours: quality ~110 min, then diversity ~65 min)


In [ ]:
import os
import shutil
import subprocess
import sys
import threading

THESIS_ROOT = "/content/Final-Thesis-Setup"
DMO_ROOT = "/content/Shonget-DMOSpeech2"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/results"
DRIVE_DIR = "/content/drive/MyDrive/exp031_snr_toh_method_comparison"
RUNNER = f"{THESIS_ROOT}/code/colab/run_method_comparison.py"
DEMAND = "/content/demand"
STAGE = str(globals().get("STAGE", "all")).strip().lower()  # set STAGE in the setup cell (Step 1), not here
assert STAGE in ("quality", "diversity", "all"), f"STAGE must be 'quality', 'diversity' or 'all', not {STAGE!r}"
print("Stage:", STAGE, {"all": "(about 3 hours)", "quality": "(about 110 min)", "diversity": "(about 65 min)"}[STAGE])

def sync_to_drive():
    if not os.path.exists(OUT_DIR):
        return
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            src = os.path.join(root, fn)
            if not os.path.exists(src) or fn.startswith("tmp_"):
                continue
            rel = os.path.relpath(src, OUT_DIR)
            dst = os.path.join(DRIVE_DIR, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)

stop_evt = threading.Event()
def sync_loop():
    while not stop_evt.wait(60):
        try:
            sync_to_drive()
        except Exception as exc:
            print("Drive sync warning:", exc)

t = threading.Thread(target=sync_loop, daemon=True)
t.start()
def ram_gb():
    info = dict(l.split(":", 1) for l in open("/proc/meminfo") if ":" in l)
    return int(info["MemTotal"].split()[0]) / 2**20, int(info["MemAvailable"].split()[0]) / 2**20

total, avail = ram_gb()
print(f"RAM: {avail:.1f} GB available of {total:.1f} GB")
if avail < 10:
    print("WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.")
LEAN = {"MALLOC_ARENA_MAX": "2", "PYTHONUNBUFFERED": "1"}  # fewer allocator arenas while the large models load; results are unaffected

os.chdir(THESIS_ROOT)
code = None
try:
    proc = subprocess.Popen([sys.executable, "-u", RUNNER, "run", "--dmo_root", DMO_ROOT, "--demand_dir", DEMAND, "--out_dir", OUT_DIR, "--stage", STAGE],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN})
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
finally:
    stop_evt.set()
    t.join(timeout=5)
    sync_to_drive()
if code in (-9, 137):
    raise SystemExit("The run was killed (exit code -9): the machine ran out of RAM. Use Runtime > Disconnect and delete runtime, "
                     "then Run all: Step 3 restores the finished cases from Drive and the run resumes from them.")
assert code == 0, f"Run failed with code {code}"
print("Run finished and synced to Drive.")

## 7. Analyze & Export Results ZIP

Runs the full analysis once `quality.csv` (180 rows) and `diversity.csv` (20 rows) are complete, otherwise a partial one. The zip is saved to Drive first, then downloaded; if the browser tab has lost its connection, take the zip from Drive.


In [ ]:
import csv
import os
import shutil
import subprocess
import sys
import zipfile
from google.colab import files

THESIS_ROOT = "/content/Final-Thesis-Setup"
OUT_DIR = f"{THESIS_ROOT}/research/experiments/EXP-031-snr-toh-method-comparison/results"
DRIVE_DIR = "/content/drive/MyDrive/exp031_snr_toh_method_comparison"
RUNNER = f"{THESIS_ROOT}/code/colab/run_method_comparison.py"
ZIP_NAME = "DMOSpeech2_EXP031_SNR_TOH_Method_Comparison_Results.zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

q_rows = sum(1 for _ in csv.DictReader(open(f"{OUT_DIR}/quality.csv", encoding="utf-8"))) if os.path.exists(f"{OUT_DIR}/quality.csv") else 0
d_rows = sum(1 for _ in csv.DictReader(open(f"{OUT_DIR}/diversity.csv", encoding="utf-8"))) if os.path.exists(f"{OUT_DIR}/diversity.csv") else 0
cmd = [sys.executable, RUNNER, "analyze", "--out_dir", OUT_DIR]
if q_rows < 180 or d_rows < 20:
    cmd.append("--allow_partial")
os.chdir(THESIS_ROOT)
subprocess.run(cmd, check=True)
print(open(f"{OUT_DIR}/METHOD_COMPARISON_REPORT.md", encoding="utf-8").read())

for root, _, fns in os.walk(OUT_DIR):
    for fn in fns:
        src = os.path.join(root, fn)
        if fn.startswith("tmp_"):
            continue
        rel = os.path.relpath(src, OUT_DIR)
        dst = os.path.join(DRIVE_DIR, rel)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, fns in os.walk(OUT_DIR):
        for fn in fns:
            if fn.startswith("tmp_"):
                continue
            src = os.path.join(root, fn)
            zf.write(src, arcname=os.path.relpath(src, OUT_DIR))

shutil.copy2(ZIP_PATH, os.path.join(DRIVE_DIR, ZIP_NAME))
print("Packaged:", ZIP_PATH)
print("Saved to Drive:", os.path.join(DRIVE_DIR, ZIP_NAME))
files.download(ZIP_PATH)